In [6]:
import numpy as np
import pandas as pd
import networkx as nx
import graphviz

In [7]:
FILEPATH = "../../data/WEE/"

# df_TC = pd.read_csv(f"{FILEPATH}Toy_WEEE_TCs.csv")
# df_inputs = pd.read_csv(f"{FILEPATH}Toy_WEEE_inputs.csv")
# df_flows = pd.read_csv(f"{FILEPATH}Toy_WEEE_flows.csv")
# df_composition = pd.read_csv(f"{FILEPATH}Toy_WEEE_composition.csv")
df_list_mat = pd.read_csv(f"{FILEPATH}list_materials.csv")

In [8]:
# # params = (sheet, source, target, ancestor)
# params = (
#     ('components', 'Category', 'Component'),
#     ('materials', 'Category', 'Material'),
#     ('elements_in_mat', 'Material', 'Element', 'Category'),
#     ('elements_in_comp', 'Component', 'Element', 'Category'),
# )

# attributes_dct = {
#     'WEEE_t': 'value',
#     'Share_Component': 'share',
#     'Share_Material': 'share',
#     'Share_element': 'share',
# }


In [9]:
res = []

for i in range(0,4):
    res.append(
        (
            df_list_mat.iloc[:,[i,i+1]]
            .drop_duplicates()
            .rename(columns={f'level {str(i)}': 0, f'level {str(i+1)}': 1,})
            .dropna()
        )
    )
    
res = pd.concat(res) #.sort_values(by=[0,1]).reset_index(drop=True)

res.to_csv("list_mat_graph.csv")
G = nx.from_pandas_edgelist(res, 0, 1, create_using=nx.DiGraph())

In [15]:
parent = 'undefindedAndMixedMaterials'
children = []

for target in G.nodes:
    paths = nx.all_simple_edge_paths(G, parent, target)
    if len(list(paths)) != 0:
        children += target,

subG = G.subgraph([parent] + children)
subG = nx.to_pandas_edgelist(subG)

# A = nx.nx_agraph.to_agraph(subG)  # convert to a graphviz graph
# A.draw("k5.png", prog="dot")

g = graphviz.Digraph('G', filename=f'{parent}', engine='dot')

# Create Graph Nodes and interconnecting Edges
for index, row in subG.iterrows():
    g.edge(row[0], row[1])

g.unflatten(stagger=5).view()

'undefindedAndMixedMaterials.pdf'

Icon theme "breeze" not found.
